In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import re

file_path = "/content/drive/MyDrive/NLP_final_project/news_entities_notebook3.parquet"
df = pd.read_parquet(file_path, engine="pyarrow")

print(df.shape)
print(df.columns.tolist())
df.head()

(129326, 16)
['url', 'date', 'title', 'text', 'full_text', 'relevance_score', 'ai_core_hits', 'impact_hits', 'business_hits', 'industry_hits', 'industry_text', 'industries', 'industry_count', 'org_entities_raw', 'org_entities', 'org_count']


,url,date,title,text,full_text,relevance_score,ai_core_hits,impact_hits,business_hits,industry_hits,industry_text,industries,industry_count,org_entities_raw,org_entities,org_count
0,https://boingboing.net/2024/07/01/this-ai-vide...,2024-07-01,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,This AI video of gymnastics might be the freak...,4,1,0,1,0,this ai video of gymnastics might be the freak...,[Media & Marketing],1,"[Wonderful Products (Contact Support, Werners ...","[Wonderful Products (Contact Support, Werners ...",3
1,https://boingboing.net/2024/09/18/if-using-ai-...,2024-09-22,"If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...","If using AI feels like a chore, try this - Boi...",19,6,0,0,1,if using ai feels like a chore try this boing ...,"[Media & Marketing, Technology]",2,"[AI, AI, Wonderful Products (Contact Support, ...","[Wonderful Products (Contact Support, SEO]",2
2,https://citylife.capetown/gl/uncategorized/the...,2023-11-10,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,The Road Ahead: How China's AI Foundation Mode...,19,4,3,0,1,the road ahead how china s ai foundation model...,"[Technology, Transportation & Logistics]",2,[AI Foundation Model is Shaping the Future of ...,[AI Foundation Model is Shaping the Future of ...,7
3,https://citylife.capetown/kk/uncategorized/mic...,2023-11-19,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,Microsoft and Nvidia to Empower Developers wit...,11,3,1,0,0,microsoft and nvidia to empower developers wit...,[Technology],1,"[Microsoft, Microsoft, өтуСенбі, Қала, Microso...","[Microsoft, өтуСенбі, Қала, Windows, Windows A...",9
4,https://citylife.capetown/lb/uncategorized/how...,2023-12-12,Google Releases New Chatbot Bard as a Strong C...,Google Releases New Chatbot Bard as a Strong C...,Google Releases New Chatbot Bard as a Strong C...,18,6,0,0,0,google releases new chatbot bard as a strong c...,[],0,"[Google, OpenAI, Google, OpenAI, Wiessel un In...","[Google, OpenAI, Wiessel un InhaltDi, Stad Lie...",8


In [3]:
print(type(df["industries"].iloc[0]))
print(type(df["org_entities"].iloc[0]))

<class 'numpy.ndarray'>
<class 'numpy.ndarray'>


In [4]:
THEME_DICT = {
    "Automation": [
        "automation", "automate", "automated", "replace manual",
        "process automation", "robotic process automation", "rpa"
    ],
    "Augmentation / Assistance": [
        "copilot", "assistant", "augment", "augmentation",
        "support workers", "help employees", "assist", "workflow support"
    ],
    "Productivity / Efficiency": [
        "productivity", "efficiency", "faster", "speed up",
        "time savings", "streamline", "optimize", "throughput"
    ],
    "Cost Reduction": [
        "cost reduction", "cost savings", "reduce costs", "lower costs",
        "cheaper", "operating margin", "expense reduction"
    ],
    "Job Displacement / Labor Impact": [
        "layoff", "layoffs", "job loss", "displacement", "replace workers",
        "replace jobs", "labor market", "workforce reduction", "headcount"
    ],
    "Workflow Redesign": [
        "workflow", "workflow redesign", "business process", "operating model",
        "redesign", "transformation", "process change"
    ],
    "Customer Service / Personalization": [
        "customer service", "chatbot", "personalization", "recommendation",
        "customer support", "call center", "consumer experience"
    ],
    "Decision Support / Analytics": [
        "decision support", "forecasting", "prediction", "analytics",
        "insight generation", "business intelligence", "risk scoring"
    ],
    "Cybersecurity / Fraud": [
        "cybersecurity", "fraud", "scam", "threat detection",
        "security monitoring", "fraud detection", "deepfake risk"
    ],
    "Regulation / Governance / Risk": [
        "regulation", "governance", "compliance", "risk",
        "responsible ai", "safety", "oversight", "policy"
    ],
    "Content Generation / Creative Work": [
        "content generation", "generate content", "image generation",
        "video generation", "creative tools", "copywriting", "design"
    ],
    "Software / Coding Assistance": [
        "code generation", "coding assistant", "software development",
        "developer productivity", "copilot", "programming assistance"
    ],
    "Healthcare / Drug Discovery": [
        "drug discovery", "clinical decision", "medical imaging",
        "radiology", "diagnosis", "patient care", "biotech research"
    ]
}

In [5]:
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["theme_text"] = df["full_text"].apply(normalize_text)

In [7]:
def get_matching_themes(text, theme_dict):
    matches = []
    for theme, keywords in theme_dict.items():
        for kw in keywords:
            pattern = r"\b" + re.escape(kw.lower()) + r"\b"
            if re.search(pattern, text):
                matches.append(theme)
                break
    return matches

df["impact_themes"] = df["theme_text"].apply(lambda x: get_matching_themes(x, THEME_DICT))
df["theme_count"] = df["impact_themes"].apply(len)

df[["title", "impact_themes", "theme_count"]].head(10)

,title,impact_themes,theme_count
0,This AI video of gymnastics might be the freak...,"[Decision Support / Analytics, Cybersecurity /...",3
1,"If using AI feels like a chore, try this - Boi...","[Augmentation / Assistance, Decision Support /...",3
2,The Road Ahead: How China's AI Foundation Mode...,[Decision Support / Analytics],1
3,Microsoft and Nvidia to Empower Developers wit...,[Productivity / Efficiency],1
4,Google Releases New Chatbot Bard as a Strong C...,"[Customer Service / Personalization, Software ...",2
5,Zoom Expands AI Offering with AI Companion and...,"[Augmentation / Assistance, Productivity / Eff...",3
6,Pro-AI Thinking: Enhancing Industrial Environm...,"[Automation, Productivity / Efficiency, Regula...",3
7,Best AI Prompts for Business Risk Management,"[Automation, Augmentation / Assistance, Produc...",5
8,Bullfrog AI Holdings Inc. [BFRG] Revenue clock...,[Cost Reduction],1
9,IIM Ahmedabad student writes project using Cha...,"[Productivity / Efficiency, Cybersecurity / Fr...",4


In [8]:
print("No-theme articles:", (df["theme_count"] == 0).sum())
print("Pct no-theme:", round((df["theme_count"] == 0).mean() * 100, 2), "%")

df["theme_count"].value_counts().sort_index()

No-theme articles: 14507
Pct no-theme: 11.22 %


,count
theme_count,
0,14507
1,27901
2,28809
3,22383
4,15471
5,9918
6,5742
7,2723
8,1261


In [9]:
df_themes = df.explode("impact_themes").copy()
df_themes = df_themes[df_themes["impact_themes"].notna()]

theme_summary = (
    df_themes.groupby("impact_themes")
    .size()
    .reset_index(name="article_count")
    .sort_values("article_count", ascending=False)
)

theme_summary.head(20)

,impact_themes,article_count
10,Regulation / Governance / Risk,87620
9,Productivity / Efficiency,47907
2,Content Generation / Creative Work,28885
4,Customer Service / Personalization,27844
1,Automation,27099
6,Decision Support / Analytics,25157
0,Augmentation / Assistance,25034
12,Workflow Redesign,20066
5,Cybersecurity / Fraud,16302
11,Software / Coding Assistance,9204


In [10]:
df[["title", "impact_themes"]].sample(15, random_state=42)

,title,impact_themes
89071,Alida Launches Industry-First AI Assistant for...,"[Augmentation / Assistance, Productivity / Eff..."
123265,Suhel Seth on Future Impact of Artificial Inte...,[]
62387,ASTERRA's new satellite based PolSAR technolog...,[Decision Support / Analytics]
4270,Where federal AI is headed next | Federal News...,"[Augmentation / Assistance, Cybersecurity / Fr..."
56300,Exotel launches AI chatbot builder ‘ExoMind’ p...,"[Job Displacement / Labor Impact, Workflow Red..."
27153,Subtle Medical Joins Imaging AI in Practice De...,"[Augmentation / Assistance, Productivity / Eff..."
22358,Wadhwani AI showcases Solutions for Social Goo...,"[Workflow Redesign, Regulation / Governance / ..."
76152,Zoom and CrowdStrike stocks ride SaaS’s AI tre...,"[Automation, Augmentation / Assistance, Cybers..."
15663,Venzee AI Accelerates Intelligent Data Syndica...,"[Automation, Augmentation / Assistance, Regula..."
10415,This AI will tell you whether you're being pai...,"[Augmentation / Assistance, Customer Service /..."


In [11]:
df[df["impact_themes"].apply(lambda x: "Job Displacement / Labor Impact" in x)][
    ["date", "title", "impact_themes"]
].head(20)

,date,title,impact_themes
19,2023-04-25,The key to responsible AI development,"[Automation, Productivity / Efficiency, Job Di..."
39,2024-12-11,Trinity Life Sciences Appoints Jonathan Jenkin...,"[Productivity / Efficiency, Job Displacement /..."
72,2022-07-13,Validus Senior Living Partners With Arena Anal...,"[Job Displacement / Labor Impact, Decision Sup..."
82,2024-01-10,Health-e Law Episode 2: AI As An Aid: Emerging...,"[Job Displacement / Labor Impact, Cybersecurit..."
92,2023-06-09,Wanted: An AI job that (mostly) doesn’t exist ...,"[Automation, Job Displacement / Labor Impact, ..."
103,2023-06-08,Wanted: An AI job that (mostly) doesn’t exist ...,"[Automation, Job Displacement / Labor Impact, ..."
131,2023-12-13,How good will AI be in 2050?,"[Productivity / Efficiency, Job Displacement /..."
132,2023-12-14,Waa maxay AI ugu awoodda badan adduunka?,"[Automation, Job Displacement / Labor Impact]"
160,2025-08-06,‘Creative theft’: arts and screen bodies respo...,"[Productivity / Efficiency, Job Displacement /..."
214,2023-03-29,"Tinkering with ChatGPT, workers wonder: Will t...","[Automation, Augmentation / Assistance, Produc..."


In [12]:
df_ind_theme = df.copy()
df_ind_theme = df_ind_theme.explode("industries")
df_ind_theme = df_ind_theme.explode("impact_themes")

df_ind_theme = df_ind_theme[
    df_ind_theme["industries"].notna() &
    df_ind_theme["impact_themes"].notna()
].copy()

In [13]:
industry_theme_summary = (
    df_ind_theme.groupby(["industries", "impact_themes"])
    .size()
    .reset_index(name="article_count")
    .sort_values(["industries", "article_count"], ascending=[True, False])
)

industry_theme_summary.head(30)

,industries,impact_themes,article_count
10,Consulting & Professional Services,Regulation / Governance / Risk,12488
9,Consulting & Professional Services,Productivity / Efficiency,7657
6,Consulting & Professional Services,Decision Support / Analytics,5150
1,Consulting & Professional Services,Automation,4786
2,Consulting & Professional Services,Content Generation / Creative Work,4567
12,Consulting & Professional Services,Workflow Redesign,4442
4,Consulting & Professional Services,Customer Service / Personalization,3747
0,Consulting & Professional Services,Augmentation / Assistance,3707
5,Consulting & Professional Services,Cybersecurity / Fraud,3018
11,Consulting & Professional Services,Software / Coding Assistance,1437


In [14]:
top_themes_by_industry = (
    industry_theme_summary
    .groupby("industries", group_keys=False)
    .head(5)
    .reset_index(drop=True)
)

top_themes_by_industry.head(50)

,industries,impact_themes,article_count
0,Consulting & Professional Services,Regulation / Governance / Risk,12488
1,Consulting & Professional Services,Productivity / Efficiency,7657
2,Consulting & Professional Services,Decision Support / Analytics,5150
3,Consulting & Professional Services,Automation,4786
4,Consulting & Professional Services,Content Generation / Creative Work,4567
5,Education,Regulation / Governance / Risk,43588
6,Education,Productivity / Efficiency,20269
7,Education,Customer Service / Personalization,13358
8,Education,Decision Support / Analytics,12534
9,Education,Content Generation / Creative Work,12419


In [15]:
df_comp_theme = df.copy()
df_comp_theme = df_comp_theme.explode("org_entities")
df_comp_theme = df_comp_theme.explode("impact_themes")

df_comp_theme = df_comp_theme[
    df_comp_theme["org_entities"].notna() &
    df_comp_theme["impact_themes"].notna()
].copy()

company_theme_summary = (
    df_comp_theme.groupby(["org_entities", "impact_themes"])
    .size()
    .reset_index(name="article_count")
    .sort_values(["org_entities", "article_count"], ascending=[True, False])
)

company_theme_summary.head(30)

,org_entities,impact_themes,article_count
0,! - India Telecom News,Productivity / Efficiency,1
1,! - India Telecom News,Regulation / Governance / Risk,1
2,"""Artificial Intelligence in",Customer Service / Personalization,1
3,"""Artificial Intelligence in",Cybersecurity / Fraud,1
4,"""Artificial Intelligence in",Decision Support / Analytics,1
5,"""Artificial Intelligence in",Productivity / Efficiency,1
6,"""Artificial Intelligence in",Regulation / Governance / Risk,1
8,"""ChatGPT",Customer Service / Personalization,3
10,"""ChatGPT",Productivity / Efficiency,2
11,"""ChatGPT",Regulation / Governance / Risk,2


In [16]:
df.to_parquet(
    "/content/drive/MyDrive/NLP_final_project/news_themes_notebook4.parquet",
    engine="pyarrow",
    index=False
)

In [17]:
theme_summary.to_csv(
    "/content/drive/MyDrive/NLP_final_project/theme_summary_notebook4.csv",
    index=False
)

industry_theme_summary.to_csv(
    "/content/drive/MyDrive/NLP_final_project/industry_theme_summary_notebook4.csv",
    index=False
)

top_themes_by_industry.to_csv(
    "/content/drive/MyDrive/NLP_final_project/top_themes_by_industry_notebook4.csv",
    index=False
)

print("saved Notebook 4 outputs")

saved Notebook 4 outputs
